# HW3 — Categorical Variables & Decision Trees (2026)

- [Official homework](https://github.com/DataTalksClub/stock-markets-analytics-zoomcamp/blob/main/cohorts/2026/homework3.md)
- [User's already-executed Colab](https://colab.research.google.com/drive/1i0mKulJ-JJ4ApGKL1rwjaqHejR3wRDKd?usp=sharing)
- [Original 2026 modeling notebook](https://colab.research.google.com/github/DataTalksClub/stock-markets-analytics-zoomcamp/blob/main/03-modeling/%5B2026%5D_Module_03_Colab_Time_Series_Modeling.ipynb)

This is an **audit companion notebook, not the user's downloaded Colab**. Values below were **transcribed from user's pasted Colab output**, not independently rerun. Its later code cells require the completed official notebook variables (`new_df`, `X_train`, etc.) to be present in the **same runtime**. To publish executed code/output faithfully, upload the actual Colab `.ipynb` file in place of this companion after review.

## User-reported Q1–Q4 values

| Q | Output from executed Colab | Status |
| --- | --- | --- |
| 1 | `month_wom_October_w4`: **0.025**; 60 month-week/115 total dummies | Matches form option |
| 2 | `pred3_manual_dgs10_5`: **0.588**; `pred4`: 0.503 | Matches form option |
| 3 | **1243**, but options are **271, 571, 1171, 1571** | **BLOCKED: audit before submission** |
| 4 | Best depth **4**, test precision **0.62917** | Matches form option |

Q4 depth-specific reported test precision: 1=0.57009, 2=0.62253, 3=0.57197, 4=0.62917, 5=0.59997, 6=0.60621, 7=0.62107, 8=0.62179, 9=0.61118, 10=0.59121, 11=0.60275, 12=0.56831.

In [ ]:
# This cell validates the pasted summary without inventing a new run.
reported_depths = {1:.57009,2:.62253,3:.57197,4:.62917,5:.59997,6:.60621,7:.62107,8:.62179,9:.61118,10:.59121,11:.60275,12:.56831}
assert max(reported_depths, key=reported_depths.get) == 4
print("Q1 (reported):",0.025)
print("Q2 (reported):",0.588)
print("Q3 (reported, DOES NOT MATCH FORM):",1243)
print("Q4 (reported best depth):",max(reported_depths,key=reported_depths.get))
print("Q4 reported precision:",max(reported_depths.values()))

## Q3 — Test-only audit

Run the **official 2026 notebook first**. The code below intentionally requires the correct explicit TEST mask rather than silently inferring a split. Do not select the nearest multiple-choice option.

In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier

assert "new_df" in globals(), "First run the official course notebook in this runtime"
assert "is_positive_growth_30d_future" in new_df, "Target missing"
target="is_positive_growth_30d_future"
pred={}
for i in range(6):
    cols=[c for c in new_df.columns if c==f"pred{i}" or c.startswith(f"pred{i}_")]
    print(f"pred{i} columns:",cols)
    assert len(cols)==1, f"Expected exactly one pred{i} column"
    pred[i]=cols[0]
print("Split-indicator candidates:",
      [c for c in new_df.columns if any(k in c.lower() for k in ("split","dataset","partition","sample","subset","set"))])
print("Rows in new_df:",len(new_df))
print("TREE configuration must be max_depth=10, random_state=42 and fit on TRAIN+VALIDATION")

In [ ]:
# Define TEST_MASK from the original notebook's actual test split.
# Example only, when a 'split' column exists:
# TEST_MASK = new_df['split'].astype(str).str.upper().eq('TEST')
assert "TEST_MASK" in globals(), "Define TEST_MASK using the original notebook's TEST rows"
mask=pd.Series(TEST_MASK,index=new_df.index).astype(bool)
assert len(mask)==len(new_df) and mask.any()
y=new_df[target].astype(int)
clf_correct=new_df[pred[5]].astype(int).eq(y)
manual_incorrect=pd.concat([new_df[pred[i]].astype(int).ne(y) for i in range(5)],axis=1).all(axis=1)
print("TEST rows",int(mask.sum()))
print("TEST tree correct",int((mask & clf_correct).sum()))
print("TEST all five hand rules incorrect",int((mask & manual_incorrect).sum()))
print("Q3 corrected TEST count",int((mask & clf_correct & manual_incorrect).sum()))
print("All-dataset count",int((clf_correct & manual_incorrect).sum()))
assert int((mask & clf_correct & manual_incorrect).sum()) in (271,571,1171,1571), "Still mismatched: examine row alignment, rule definitions, fit split and TEST selection" 

## Q5 — Suggested answer

I would add VIX and changes in trading volume as new features. A higher VIX can mean more market uncertainty, which may reduce the chance of positive returns. A big change in trading volume can show stronger buying or selling activity. These features may help the model predict if a stock will have a positive return in the next 30 days.

**Methodology note:** HW3's required selection by TEST precision is not a recommended practice for independent out-of-sample evaluation. Use validation data for tuning in a real project.